# DEU Generative Relativity: Complete Analysis Notebook

This notebook recreates the summary tables and figures used in the manuscript *Generative Relativity in a Triangular DEU Causal Foam*. It is designed for GitHub release.

The notebook is self-contained: it embeds the final summary datasets from the simulation and GW150914 analysis. If full CSV outputs are placed in `data/`, the same plotting cells can be adapted to load those files instead.

Main layers:

1. Baseline finite 2+1D Lorentzian scaling regime.
2. Layer 2 generative lapse from useful-bandwidth drain.
3. Layer 3 refinement-sink null and topology-stitch throat signal.
4. Preliminary GW150914 nonlinear ringdown compression candidate.


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# This notebook can be run either from the repository root or from notebooks/.
ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'data'
FIG = ROOT / 'figures'
FIG.mkdir(exist_ok=True)

def savefig(name):
    for ext in ['png','pdf']:
        plt.savefig(FIG / f'{name}.{ext}', bbox_inches='tight', dpi=220)
    plt.show()

print('ROOT =', ROOT)
print('DATA =', DATA)
print('FIG =', FIG)


## Baseline cap sweep

In [ ]:

cap_sweep = pd.read_csv(DATA / 'baseline_cap_sweep.csv')
display(cap_sweep)
plt.figure(figsize=(7,4.5))
for col in ['dMM','V_slope','Amid_slope','Amax_slope']:
    plt.plot(cap_sweep['cap'], cap_sweep[col], marker='o', label=col)
plt.axhline(3, linestyle='--', linewidth=1, label='spacetime target 3')
plt.axhline(2, linestyle=':', linewidth=1, label='spatial exponent target 2')
plt.xlabel('scheduler cap')
plt.ylabel('dimension / exponent')
plt.title('Baseline 2+1D scaling diagnostics')
plt.legend(frameon=False, fontsize=8)
plt.grid(True, alpha=0.3)
savefig('nb_baseline_cap_sweep')


## Metric gate and depth replay

In [ ]:

depth_summary = pd.read_csv(DATA / 'depth_summary_cap512_seed101_10k.csv')
metric_gate = pd.read_csv(DATA / 'metric_gate_summary.csv')
display(metric_gate)
display(depth_summary)
plt.figure(figsize=(7,4.5))
plt.plot(depth_summary['epoch'], depth_summary['depth_med'], marker='o', label='median refinement depth')
plt.plot(depth_summary['epoch'], depth_summary['depth_max'], marker='o', label='max refinement depth')
plt.xlabel('epoch')
plt.ylabel('refinement depth')
plt.title('Depth replay: real face-depth metric recovered')
plt.legend(frameon=False)
plt.grid(True, alpha=0.3)
savefig('nb_depth_replay_depths')
plt.figure(figsize=(7,4.5))
plt.plot(depth_summary['epoch'], depth_summary['area_sum_if_split_area_thirds'], marker='o')
plt.xlabel('epoch')
plt.ylabel('sum 3^{-k_f}')
plt.title('Refinement-weighted area conservation')
plt.grid(True, alpha=0.3)
savefig('nb_area_conservation')


## Layer 2 useful lapse

In [ ]:

lapse_summary = pd.read_csv(DATA / 'layer2_useful_lapse_summary.csv')
display(lapse_summary)
plt.figure(figsize=(7,4.5))
for run, df in lapse_summary.groupby('run'):
    plt.errorbar(df['m_defects'], df['Omega_useful_rate_mean'], yerr=df['Omega_useful_rate_sem'], marker='o', capsize=3, label=run)
plt.xlabel('defect load m')
plt.ylabel('useful lapse Omega_useful')
plt.title('Layer 2: useful generative lapse rises with defect load')
plt.legend(frameon=False, fontsize=8)
plt.grid(True, alpha=0.3)
savefig('nb_layer2_useful_lapse')


## Layer 2 radial lapse residual

In [ ]:

radial_lapse = pd.read_csv(DATA / 'layer2_radial_lapse_residual.csv')
display(radial_lapse)
plt.figure(figsize=(7,4.5))
for window in ['inner_0p15_0p45','mid_0p45_0p90','outer_0p90_1p50','wide_0p15_1p50']:
    df = radial_lapse[radial_lapse['radial_window']==window]
    plt.plot(df['m_defects'], df['Omega_update_residual_mean'], marker='o', label=window)
plt.axhline(0, linewidth=1)
plt.xlabel('defect load m')
plt.ylabel('residual update lapse')
plt.title('Radial lapse residuals')
plt.legend(frameon=False, fontsize=8)
plt.grid(True, alpha=0.3)
savefig('nb_radial_lapse_residual')


## Layer 3 refinement-sink null

In [ ]:

refinement_null = pd.read_csv(DATA / 'layer3_refinement_sink_null.csv')
display(refinement_null)
plt.figure(figsize=(7,4.5))
for window in refinement_null['radial_window'].unique():
    df = refinement_null[refinement_null['radial_window']==window]
    plt.errorbar(df['m_defects'], df['median_delta_C_mean'], yerr=df['median_delta_C_sem'], marker='o', capsize=3, label=window)
plt.axhline(0, linewidth=1)
plt.xlabel('defect load m')
plt.ylabel('Delta C = C0 - Cm')
plt.title('Refinement sink null: no robust circumference pinch')
plt.legend(frameon=False, fontsize=8)
plt.grid(True, alpha=0.3)
savefig('nb_refinement_sink_null_deltaC')


## Layer 3 q-sector positive control

In [ ]:

qcontrol = pd.read_csv(DATA / 'layer3_qsector_positive_control.csv')
display(qcontrol)
plt.figure(figsize=(7,4.5))
plt.plot(qcontrol['q_sectors'], qcontrol['expected_C_ratio_vs_q6'], marker='o', label='expected q/6')
plt.plot(qcontrol['q_sectors'], qcontrol['measured_C_ratio_inner'], marker='x', label='measured inner')
plt.xlabel('sector count q')
plt.ylabel('C_q/C_6')
plt.title('Positive control: explicit q-sector holonomy')
plt.legend(frameon=False)
plt.grid(True, alpha=0.3)
savefig('nb_qsector_positive_control')


## Layer 3 exterior-boundary throat

In [ ]:

exterior_boundary = pd.read_csv(DATA / 'layer3_exterior_boundary_throat.csv')
throat_verdict = pd.read_csv(DATA / 'layer3_exterior_boundary_throat_verdict.csv')
display(throat_verdict)
display(exterior_boundary)
mid = exterior_boundary[exterior_boundary['radial_window']=='mid_0p60_1p00']
plt.figure(figsize=(7,4.5))
plt.plot(mid['m_defects'], mid['median_delta_C_mean'], marker='o')
plt.axhline(0, linewidth=1)
plt.xlabel('stitch defect load m')
plt.ylabel('exterior-boundary Delta C')
plt.title('Topology stitch: exterior-boundary circumference pinch')
plt.grid(True, alpha=0.3)
savefig('nb_exterior_boundary_deltaC_mid')
plt.figure(figsize=(7,4.5))
plt.plot(mid['m_defects'], mid['median_delta_A_over_R2_mean'], marker='o')
plt.axhline(0, linewidth=1)
plt.xlabel('stitch defect load m')
plt.ylabel('exterior-boundary Delta A/R^2')
plt.title('Topology stitch: throat area response')
plt.grid(True, alpha=0.3)
savefig('nb_exterior_boundary_deltaA_mid')


## LIGO mode ablation

In [ ]:

ligo_ablation = pd.read_csv(DATA / 'ligo_mode_ablation_summary.csv')
display(ligo_ablation)
plt.figure(figsize=(7,4.5))
plt.plot(ligo_ablation['n_modes'], ligo_ablation['delta_BIC_DEU_vs_GR'], marker='o', label='DEU vs GR')
plt.plot(ligo_ablation['n_modes'], ligo_ablation['delta_BIC_DEU_vs_UNIFORM'], marker='o', label='DEU vs uniform')
plt.plot(ligo_ablation['n_modes'], ligo_ablation['delta_BIC_DEU_vs_FREE'], marker='o', label='DEU vs free scale')
plt.axhline(0, linewidth=1)
plt.axhline(6, linestyle='--', linewidth=1, label='BIC 6')
plt.xlabel('number of tones included')
plt.ylabel('Delta BIC in favor of DEU')
plt.title('GW150914 toy ringdown: mode-ablation compression test')
plt.legend(frameon=False, fontsize=8)
plt.grid(True, alpha=0.3)
savefig('nb_ligo_mode_ablation')


## LIGO detector consistency

In [ ]:

detector_consistency = pd.read_csv(DATA / 'ligo_detector_consistency.csv')
family_summary = pd.read_csv(DATA / 'ligo_family_lookelsewhere_summary.csv')
display(family_summary)
display(detector_consistency)
plt.figure(figsize=(6,4.5))
plt.bar(detector_consistency['ifo'], detector_consistency['delta_BIC_DEU_vs_GR'])
plt.axhline(0, linewidth=1)
plt.xlabel('interferometer')
plt.ylabel('Delta BIC DEU vs GR')
plt.title('Detector consistency for three-tone DEU candidate')
plt.grid(True, axis='y', alpha=0.3)
savefig('nb_ligo_detector_consistency')


## Reanalysis hooks

For full reproduction, place native CSV outputs from the DEU notebooks and the GW150914 search under `data/raw/`, then replace the embedded summaries above with grouped aggregations from those CSVs. The final paper should distinguish embedded-summary reproduction from full raw-run reproduction.
